In [2]:
# Modülde neler yapılacağını bul
# Trust İndex
# Performans İndex 
# WOEleyip portföy oluşturma

In [3]:
# QuiverQuant API key'i dosyadan oku
with open("Quiver API KEY.txt", "r") as f:
    API_KEY = f.read().strip()

headers = {"Authorization": f"Token {API_KEY}"}


In [ ]:
import pandas as pd
import requests

def list_slickcharts_sp500() -> pd.DataFrame:
    # Ref: https://stackoverflow.com/a/75845569/
    url = 'https://www.slickcharts.com/sp500'
    user_agent = 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/111.0'  # Default user-agent fails.
    response = requests.get(url, headers={'User-Agent': user_agent})
    return pd.read_html(response.text, match='Symbol', index_col='Symbol')[0]
sp500tickers = list_slickcharts_sp500().index.tolist()

/var/folders/97/ch83kwyd58z8krz3tltkbxkm0000gn/T/ipykernel_17471/1385490009.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  return pd.read_html(response.text, match='Symbol', index_col='Symbol')[0]


In [ ]:
headers = {"Authorization": f"Bearer {API_KEY}"}
base_url = "https://api.quiverquant.com/beta/historical/congresstrading/"
r = requests.get(base_url + "NVDA", headers=headers)
data = r.json()
df = pd.DataFrame(data)


=== NVDA ===
       Representative BioGuideID  ReportDate TransactionDate Ticker  \
0       Valerie Hoyle    H001094  2025-10-10      2025-09-23   NVDA   
1           Ro Khanna    K000389  2025-10-03      2025-09-05   NVDA   
2  Sheldon Whitehouse    W000802  2025-10-03      2025-09-04   NVDA   

      Transaction               Range            House   Amount Party  \
0            Sale  $50,001 - $100,000  Representatives  50001.0     D   
1            Sale    $1,001 - $15,000           Senate   1001.0     D   
2  Sale (Partial)   $15,001 - $50,000           Senate  15001.0     D   

  last_modified TickerType Description  ExcessReturn  PriceChange  SPYChange  
0    2025-10-10         ST        None      4.300788     7.319397   3.018609  
1    2025-10-06       None        None      8.761468    14.650940   5.889472  
2    2025-10-03      Stock        None      5.951744    11.551905   5.600161  

=== MSFT ===
         Representative BioGuideID  ReportDate TransactionDate Ticker  \
0    

In [14]:
from time import sleep

base_url = "https://api.quiverquant.com/beta/historical/congresstrading/"
all_data = []

for i, ticker in enumerate(sp500tickers):
    url_t = base_url + ticker
    r = requests.get(url_t, headers=headers)

    if r.status_code == 200:
        try:
            data = r.json()
            if data:
                for row in data:
                    all_data.append({
                        "Ticker": ticker,
                        "Representative": row.get("Representative"),
                        "Party": row.get("Party"),
                        "House": row.get("House"),
                        "BioGuideID": row.get("BioGuideID")
                    })
        except Exception as e:
            print("JSON hatası:", ticker, e)
    else:
        print("Hata:", ticker, r.status_code)
    sleep(0.5)

print(f"\nToplam {len(all_data)} işlem kaydı çekildi.")
# === 4) DataFrame oluştur ===
df = pd.DataFrame(all_data)

# === 5) Unique temsilciler ===
distinct_df = df.drop_duplicates(subset=["Representative", "Party", "House", "BioGuideID"]).reset_index(drop=True)

print("\n=== Unique Congress Traders ===")
print(distinct_df.head(20))

Hata: CRWD 429
Hata: PLD 500
Hata: COP 500
Hata: ABNB 500
Hata: RCL 500
Hata: PWR 500
Hata: CMG 500
Hata: LHX 500
Hata: AMP 500
Hata: A 500
Hata: PEG 500
Hata: OXY 500
Hata: FICO 500
Hata: YUM 500
Hata: VMC 500
Hata: MLM 500
Hata: CBOE 429
Hata: BIIB 500
Hata: TSN 500
Hata: SWKS 429
Hata: ALB 429
Hata: RVTY 429
Hata: DAY 429
Hata: MOS 500
Hata: CPB 500

Toplam 55907 işlem kaydı çekildi.

=== Unique Congress Traders ===
   Ticker          Representative Party            House BioGuideID
0    NVDA           Valerie Hoyle     D  Representatives    H001094
1    NVDA               Ro Khanna     D           Senate    K000389
2    NVDA      Sheldon Whitehouse     D           Senate    W000802
3    NVDA             Cleo Fields     D  Representatives    F000110
4    NVDA       Michael T. McCaul     R           Senate    M001157
5    NVDA            John Boozman     R           Senate    B001236
6    NVDA              Angus King     I           Senate    K000383
7    NVDA         Josh Gottheimer

In [20]:
distinct_df[distinct_df["Representative"] == "Tim Moore"]

,Ticker,Representative,Party,House,BioGuideID
151,AAPL,Tim Moore,R,Representatives,M001236


In [30]:
# === 6) Kaydet ===
# Tüm işlemleri (ham veri)
df.to_parquet("congress_all_data.parquet", index=False)

# Unique temsilciler (kimler işlem yapmış)
distinct_df.to_parquet("congress_distinct_traders.parquet", index=False)

print("\n✅ Veriler başarıyla kaydedildi:")
print(" - congress_all_data.parquet")
print(" - congress_distinct_traders.parquet")



✅ Veriler başarıyla kaydedildi:
 - congress_all_data.parquet
 - congress_distinct_traders.parquet


In [32]:
import pandas as pd

# === 1) Daha önce kaydettiğin ham veriyi oku ===
df = pd.read_parquet("congress_all_data.parquet")

print(f"Toplam {len(df):,} işlem kaydı yüklendi.")
print(f"Distinct ticker sayısı: {df['Ticker'].nunique()}")
print(f"Distinct temsilci sayısı: {df['Representative'].nunique()}\n")

# === 2) Senatör / temsilci bazlı arama ===
# İsimle:
target_names = ["Nancy Pelosi", "Dan Crenshaw"]  # örnek
by_name = df[df["Representative"].isin(target_names)].copy()

# BioGuideID ile:
target_ids = ["P000197", "C001084"]  # örnek
by_id = df[df["BioGuideID"].isin(target_ids)].copy()

# === 3) Kimler işlem yapmış (distinct liste) ===
distinct_people = (
    df[["Representative", "Party", "House", "BioGuideID"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)
print("\n=== Distinct Congress Traders ===")
print(distinct_people.head(20))

# === 4) Belirli bir kişi için özet ===
one = by_id if not by_id.empty else by_name
if not one.empty:
    print("\nKişinin işlem yaptığı distinct tickerlar ve adetleri:")
    print(one.groupby("Ticker").size().sort_values(ascending=False).head(20))
else:
    print("\nSeçilen kişi(ler) için veri bulunamadı.")

# === 5) İstersen kişi bazında toplam trade sayısı tablosu ===
top_traders = (
    df.groupby(["Representative", "Party", "House"])
    .size()
    .reset_index(name="TradeCount")
    .sort_values("TradeCount", ascending=False)
)
print("\nEn çok işlem yapan 10 kişi:")
print(top_traders.head(10))


Toplam 55,907 işlem kaydı yüklendi.
Distinct ticker sayısı: 478
Distinct temsilci sayısı: 330


=== Distinct Congress Traders ===
            Representative Party            House BioGuideID
0            Valerie Hoyle     D  Representatives    H001094
1                Ro Khanna     D           Senate    K000389
2       Sheldon Whitehouse     D           Senate    W000802
3              Cleo Fields     D  Representatives    F000110
4        Michael T. McCaul     R           Senate    M001157
5             John Boozman     R           Senate    B001236
6               Angus King     I           Senate    K000383
7          Josh Gottheimer     D  Representatives    G000583
8             Lisa Mcclain     R  Representatives    M001136
9            Rob Bresnahan     R  Representatives    B001327
10  Marjorie Taylor Greene     R  Representatives    G000596
11        Jefferson Shreve     R  Representatives    S001229
12         Bruce Westerman     R  Representatives    W000821
13         Jared

In [33]:
top_traders

,Representative,Party,House,TradeCount
266,Ro Khanna,D,Senate,18728
195,Michael T. McCaul,R,Senate,7071
147,Josh Gottheimer,D,Representatives,1895
67,David Perdue,R,Senate,971
171,Lois Frankel,D,Representatives,868
...,...,...,...,...
292,Stephanie Bice,R,Representatives,1
294,Stephen F. Lynch,D,Representatives,1
56,Dave Min,D,Representatives,1
137,John Fetterman,D,Senate,1
